# Career Orientation — Data Generation

Generate 1200 synthetic student profiles (200 per career branch) using Gemini 2.5 Flash.

**Steps:**
1. Setup & test API connection
2. Generate 1 test sample → `data/test.json`
3. Full generation → `data/raw_profiles.json`

## Cell 1 — Install & Imports

In [8]:
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'google-generativeai', 'python-dotenv'
], check=True)
print('Packages installed.')

Packages installed.


In [9]:
import os
import json
import time
import re
from pathlib import Path

import google.generativeai as genai
from dotenv import load_dotenv

# ── Paths (relative to notebooks/) ───────────────────────────────────────────
DATA_DIR = Path('..') / 'data'
DATA_DIR.mkdir(exist_ok=True)

RAW_DATA_PATH  = DATA_DIR / 'raw_profiles.json'
TEST_DATA_PATH = DATA_DIR / 'test.json'

# ── Load API key ──────────────────────────────────────────────────────────────
env_path = Path('..') / '.env'
load_dotenv(dotenv_path=env_path)
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')
if not GEMINI_API_KEY:
    raise EnvironmentError(f'GEMINI_API_KEY not found. Create {env_path.resolve()}')
genai.configure(api_key=GEMINI_API_KEY)

gemini_model = genai.GenerativeModel('gemini-2.5-flash')

# ── Constants ─────────────────────────────────────────────────────────────────
CATEGORIES = [
    'Software Engineering', 'Data Science', 'AI/ML',
    'Web Development', 'Business Analytics', 'UX/UI Design',
]
PROFILES_PER_CLASS = 200
BATCH_SIZE_FULL    = 15

print('Setup complete.')

Setup complete.


## Cell 2 — Helper Functions

In [10]:
REQUIRED_FIELDS   = {'skills', 'interests', 'academic_performance', 'projects', 'goals', 'label'}
REQUIRED_ACADEMIC = {'math', 'sciences', 'languages', 'arts'}
REQUIRED_GOALS    = {'salary_expectation', 'remote_preference', 'target_field'}


def strip_markdown(text: str) -> str:
    """Remove ```json ... ``` fences that Gemini sometimes adds."""
    text = text.strip()
    text = re.sub(r'^```(?:json)?\s*', '', text)
    text = re.sub(r'\s*```$', '', text)
    return text.strip()


def validate_profile(profile: dict, category: str) -> list[str]:
    """Validate a single profile. Returns list of errors (empty = valid)."""
    errors = []
    missing = REQUIRED_FIELDS - set(profile.keys())
    if missing:
        return [f'Missing fields: {missing}']

    if not isinstance(profile['skills'], list) or len(profile['skills']) == 0:
        errors.append("'skills' must be a non-empty list")
    if not isinstance(profile['interests'], list) or len(profile['interests']) == 0:
        errors.append("'interests' must be a non-empty list")
    if not isinstance(profile['projects'], list) or len(profile['projects']) == 0:
        errors.append("'projects' must be a non-empty list")

    acad = profile.get('academic_performance', {})
    if not isinstance(acad, dict):
        errors.append("'academic_performance' must be a dict")
    else:
        missing_acad = REQUIRED_ACADEMIC - set(acad.keys())
        if missing_acad:
            errors.append(f'Missing academic fields: {missing_acad}')
        for subj in REQUIRED_ACADEMIC & set(acad.keys()):
            score = acad[subj]
            if not isinstance(score, (int, float)) or score < 0 or score > 20:
                errors.append(f'academic_performance.{subj} = {score} — must be 0-20')

    goals = profile.get('goals', {})
    if not isinstance(goals, dict):
        errors.append("'goals' must be a dict")
    else:
        missing_goals = REQUIRED_GOALS - set(goals.keys())
        if missing_goals:
            errors.append(f'Missing goals fields: {missing_goals}')

    return errors


def generate_batch(category: str, batch_num: int, count: int = 15) -> list[dict]:
    """Call Gemini to generate `count` profiles. Raises on error."""
    # Rate limit: 30s between calls (Gemini Flash free tier ~15-20 RPM)
    print('  ... waiting 30s for rate limit ...')
    time.sleep(30)
    
    prompt = f"""
You are generating a DIVERSE synthetic dataset for a career orientation classifier.

Task: Generate exactly {count} student profiles oriented toward the career: **{category}**.
This is batch {batch_num} — profiles MUST be noticeably different from previous batches.

Diversity requirements:
- Vary skill level: beginner / intermediate / advanced
- Vary background: self-taught / university / bootcamp / vocational
- Vary age: between 18 and 28
- Vary geographic region: Europe, Africa, Southeast Asia, Latin America, North America, Middle East
- Vary academic strengths and weaknesses realistically

Return a JSON array of {count} objects. Each object must have EXACTLY these fields:
{{
  "skills": ["skill1", ...],
  "interests": ["interest1", ...],
  "academic_performance": {{ "math": 0-20, "sciences": 0-20, "languages": 0-20, "arts": 0-20 }},
  "projects": ["desc1", ...],
  "goals": {{ "salary_expectation": "<value>", "remote_preference": "<value>", "target_field": "<value>" }},
  "label": "{category}"
}}

Return ONLY the raw JSON array. No markdown fences, no explanation.
"""
    response = gemini_model.generate_content(prompt)
    raw = strip_markdown(response.text)
    profiles = json.loads(raw)

    if not isinstance(profiles, list):
        raise ValueError(f'Expected JSON array, got {type(profiles).__name__}')
    if len(profiles) != count:
        raise ValueError(f'Expected {count} profiles, got {len(profiles)}')

    for i, p in enumerate(profiles):
        p['label'] = category
        errs = validate_profile(p, category)
        if errs:
            raise ValueError(f'Profile {i} invalid: {"; ".join(errs)}')

    return profiles

print('Helper functions defined.')

Helper functions defined.


## Cell 3 — Test Run (1 sample)

Run this first! Generates 1 profile and saves to `data/test.json`.  
If this fails, do NOT run the full generation — check your API key and error message.

In [4]:
category = CATEGORIES[0]  # Software Engineering
print(f'[TEST] Generating 1 profile for "{category}"...')

try:
    profiles = generate_batch(category, batch_num=1, count=1)
    profile = profiles[0]
    with open(TEST_DATA_PATH, 'w', encoding='utf-8') as f:
        json.dump(profile, f, ensure_ascii=False, indent=2)
    print(f'[TEST] PASSED — saved to {TEST_DATA_PATH}')
    print(json.dumps(profile, indent=2, ensure_ascii=False))
except json.JSONDecodeError as e:
    print(f'[TEST] FAILED — Gemini returned invalid JSON: {e}')
except ValueError as e:
    print(f'[TEST] FAILED — Validation error: {e}')
except Exception as e:
    print(f'[TEST] FAILED — {type(e).__name__}: {e}')

[TEST] Generating 1 profile for "Software Engineering"...
[TEST] PASSED — saved to ..\data\test.json
{
  "skills": [
    "Python",
    "Java",
    "Data Structures",
    "Algorithms",
    "Git",
    "SQL",
    "Flask",
    "JavaScript (basics)",
    "HTML/CSS"
  ],
  "interests": [
    "Competitive programming",
    "Open-source contributions",
    "Learning new frameworks",
    "Solving complex problems",
    "Tech meetups"
  ],
  "academic_performance": {
    "math": 18,
    "sciences": 17,
    "languages": 13,
    "arts": 8
  },
  "projects": [
    "Developed a RESTful API for a small e-commerce platform using Python, Flask, and PostgreSQL.",
    "Contributed to an open-source data processing library, focusing on bug fixes and feature enhancements.",
    "Implemented a file management utility in Java, including features for sorting and searching."
  ],
  "goals": {
    "salary_expectation": "€45,000 - €55,000",
    "remote_preference": "Hybrid",
    "target_field": "Backend Developm

## Cell 4 — Full Generation (1200 profiles)

**Only run this after the test above passes!**

- 6 categories x 14 batches = 84 API calls
- 3 retries per failed batch
- Partial progress saved after each category to `raw_profiles_partial.json`

In [ ]:
from collections import Counter

all_profiles = []
partial_path = DATA_DIR / 'raw_profiles_partial.json'
max_retries = 5
start_time = time.time()

# Resume from partial file OR existing raw_profiles.json
resume_path = partial_path if partial_path.exists() else (RAW_DATA_PATH if RAW_DATA_PATH.exists() else None)
if resume_path:
    print(f'Resuming from: {resume_path}')
    with open(resume_path, 'r', encoding='utf-8') as f:
        all_profiles = json.load(f)
    print(f'Loaded {len(all_profiles)} profiles.')

# Count profiles per category to find what's missing
category_counts = Counter(p['label'] for p in all_profiles)
print('Current counts:', dict(category_counts))

for cat_idx, category in enumerate(CATEGORIES):
    existing = category_counts.get(category, 0)
    needed = PROFILES_PER_CLASS - existing

    if needed <= 0:
        print(f'\n[{cat_idx + 1}/{len(CATEGORIES)}] SKIPPING {category} ({existing}/{PROFILES_PER_CLASS} complete)')
        continue

    print(f'\n[{cat_idx + 1}/{len(CATEGORIES)}] {category} — need {needed} more (have {existing})')

    # Build batch list for remaining profiles
    full_batches = needed // BATCH_SIZE_FULL
    remainder = needed % BATCH_SIZE_FULL
    # batch_num starts after existing batches for diversity prompt
    existing_batches = existing // BATCH_SIZE_FULL
    batches = [(existing_batches + i + 1, BATCH_SIZE_FULL) for i in range(full_batches)]
    if remainder > 0:
        batches.append((existing_batches + full_batches + 1, remainder))

    total_batches = len(batches)
    cat_new = 0

    for i, (batch_num, count) in enumerate(batches):
        success = False
        for attempt in range(1, max_retries + 1):
            try:
                # On last retry, use smaller batch size to reduce JSON errors
                actual_count = min(count, 5) if attempt == max_retries and count > 5 else count
                batch = generate_batch(category, batch_num, actual_count)
                cat_new += len(batch)
                all_profiles.extend(batch)
                print(f'  batch {i + 1:02d}/{total_batches} — {len(batch)} profiles (new: {cat_new}/{needed})')
                success = True

                # Save partial progress after EVERY successful batch
                with open(partial_path, 'w', encoding='utf-8') as f:
                    json.dump(all_profiles, f, ensure_ascii=False, indent=2)
                break
            except json.JSONDecodeError as e:
                print(f'  batch {i + 1:02d} attempt {attempt}/{max_retries} — bad JSON: {e}')
            except ValueError as e:
                print(f'  batch {i + 1:02d} attempt {attempt}/{max_retries} — validation: {e}')
            except Exception as e:
                err_msg = str(e).lower()
                if 'rate' in err_msg or '429' in err_msg or 'quota' in err_msg:
                    print(f'  batch {i + 1:02d} attempt {attempt}/{max_retries} — RATE LIMITED, waiting 90s ...')
                    time.sleep(90)
                else:
                    print(f'  batch {i + 1:02d} attempt {attempt}/{max_retries} — {type(e).__name__}: {e}')

            if attempt < max_retries:
                backoff = 10 * (2 ** (attempt - 1))  # 10s, 20s, 40s, 80s
                print(f'    retrying in {backoff}s ...')
                time.sleep(backoff)

        if not success:
            print(f'  WARNING: batch {batch_num} failed after {max_retries} attempts — skipping')

    print(f'  => {category}: +{cat_new} profiles (total: {category_counts.get(category, 0) + cat_new})')

# Final save
with open(RAW_DATA_PATH, 'w', encoding='utf-8') as f:
    json.dump(all_profiles, f, ensure_ascii=False, indent=2)
if partial_path.exists():
    partial_path.unlink()

elapsed = time.time() - start_time
expected = PROFILES_PER_CLASS * len(CATEGORIES)
final_counts = Counter(p['label'] for p in all_profiles)
print(f'\nDone! {len(all_profiles)}/{expected} profiles saved to {RAW_DATA_PATH}')
print(f'Time elapsed: {elapsed / 60:.1f} minutes')
print('Per category:', dict(final_counts))
if len(all_profiles) < expected:
    print(f'WARNING: {expected - len(all_profiles)} profiles still missing.')